# Module 3.3 — Notebook B: Improve the baseline

In Notebook A you measured where the baseline fails. This notebook tests whether it can be fixed.

You will try three practical levers: better prompt design, few-shot examples, and data label quality. Each is tested against the same baseline so results are comparable.

The goal is not perfect scores. The goal is to understand what actually moves the metrics — and at what cost.

## What this notebook explores

This notebook focuses on **improvement**, not initial validation.

1. **Prompt refinement** — Can clearer instructions improve routing quality?
2. **Few-shot learning** — Do labeled examples help the model route better?
3. **Trade-offs** — Do improvements change cost or latency?
4. **Diagnostics** — Where does the model still fail, and why?

Assume you have completed Notebook A (or at least have a comparable baseline). You’ll use the **same dataset** and compare every change to that reference.

## Setup

Install dependencies and load helpers. Run the cell below, then move on to **Settings**.

- **Colab:** the cell installs from GitHub automatically.
- **Local:** replace the install line with `%pip install -q -e .` (from the repo root).

Keep your API key in **Secrets** (Colab) or a local **`.env`** file — never commit real keys.

In [ ]:
%pip install -q git+https://github.com/mnrozhkov/ai_leader.git
%matplotlib inline

import os

from dotenv import load_dotenv

from ai_leader import *

load_dotenv(override=False)

## Step 1 — Settings

Set your API key and experiment parameters below. The model and dataset should match what you used in Notebook A so comparisons are meaningful.

### API key & parameters

This notebook calls the **Nebius inference API** (OpenAI-compatible).
Get your key at [Nebius AI Studio](https://studio.nebius.ai) → Settings → API Keys.
Store it as a Colab Secret named `NEBIUS_API_KEY`, or add it to a local `.env` file — do **not** paste a real key into the notebook.

**401 error?** Make sure you use a Nebius *inference* API key (not an OpenAI key). See [Nebius API authentication](https://docs.nebius.com/studio/api/authentication) for regional endpoint options.

In [ ]:
api_key = setup_api_key()

# Dataset — replace with your own CSV URL or local path to use different data
DATASET_URL = os.getenv(
    "AI_LEADER_DATASET_PATH",
    "https://docs.google.com/spreadsheets/d/e/2PACX-1vSU5zvx8wgk9FMEcRGlCtXkE4_T90OgsrqU4QPNZC478Rsp5JEBEEUjvlMkY3iMoiAmpa1zQ5QFkgT5/pub?output=csv",
)

### Load the dataset

Load the **same labeled evaluation dataset** used in Notebook A (same default URL in Settings, or the same `AI_LEADER_DATASET_PATH` if you pointed it at a local `.csv`).

Reusing the same data is important: it makes baseline vs improved comparisons meaningful instead of mixing dataset effects with model/prompt effects.

In [ ]:
df = load_and_validate_dataset(DATASET_URL)
print(f"Rows loaded: {len(df)}")
df.head(3)

**Optional:** department distribution in gold labels.

## Step 2 — Baseline reference run

First, select the model you chose in Notebook A and run the baseline prompt. This is the reference point for all later comparisons.


Set `MODEL_TO_EVALUATE` to the model you selected in Notebook A (Step 4). The default below is the suggested model from the comparison run — update it if your evaluation pointed elsewhere.

Using the same model as Notebook A is important: it ensures that any accuracy change in this notebook comes from prompt or data changes, not from switching models.

In [ ]:
# Best model from Notebook A — change if your evaluation picked a different one
MODEL_TO_EVALUATE = "openai/gpt-oss-120b"
client = create_client(api_key, model=MODEL_TO_EVALUATE)

In [ ]:
baseline_run = await evaluate_model_on_dataframe_async(
    df=df,
    model=MODEL_TO_EVALUATE,
    client=client,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
)

Check the baseline results before changing anything. Note which dimensions fail — that tells you which improvement to try first.

In [ ]:
baseline_decision = evaluate_decision(baseline_run)
display_mvp_decision(decision=baseline_decision)

The baseline decision table shows where the gaps are. Note which dimensions fail before moving on — this determines which step in this notebook to focus on first.

- Quality fails → Step 3 (prompt improvement) is the highest-leverage next action.
- Safety fails → the model's confidence is poorly calibrated; Step 4 (high-confidence error inspection) is the priority.
- Cost fails → a lighter model is worth testing; this notebook does not cover model swaps, but the comparison setup from Notebook A Step 3 can be reused.
- Speed fails → check whether a smaller model from the Notebook A shortlist has lower latency before optimizing the prompt.

## Step 3 — Improve the prompt

Can a better prompt improve routing quality? We run the **improved prompt** on the same data and model, then compare it to the baseline.

Any change in accuracy, safety, or cost is attributable to the prompt alone — everything else stays fixed.

Write your improved system prompt below. The goal is to reduce the label confusion patterns you observed in Step 2 — sharper department definitions, clearer examples of what belongs in each category, and explicit instructions for edge cases. If you saved a preset in the Module 3.2 Playground, start from that.

In [ ]:
# ... edit this prompt based on the errors you found in Step 2 ...

IMPROVED_SYSTEM_PROMPT = """
You are a support routing assistant for Candlekeep Retail Group...

"""

In [ ]:
improved_prompt_run = await evaluate_model_on_dataframe_async(
    df=df,
    model=MODEL_TO_EVALUATE,
    client=client,
    system_prompt=IMPROVED_SYSTEM_PROMPT,
)

In [ ]:
improved_prompt_decision = evaluate_decision(improved_prompt_run)
display_mvp_decision(decision=improved_prompt_decision)

In [ ]:
_ = display_run_comparison_table(
    baseline=baseline_run,
    new=improved_prompt_run,
    eval_df=df,
    new_label="Improved prompt",
    show_all=False,
)

The comparison table shows the effect of the prompt change in isolation — same model, same data, different instructions.

- Focus on `delta_vs_baseline` for department accuracy and unsafe auto-route rate. These are the two dimensions most sensitive to prompt clarity.
- A longer prompt will almost always increase cost and latency slightly. An improvement is worth keeping if the accuracy gain is meaningful relative to that increase. A 1–2% accuracy gain that doubles latency is not a useful trade-off for a routing task.
- If accuracy improved but safety worsened (unsafe auto-route rate increased), the new prompt may be producing more confident but less reliable outputs — inspect the high-confidence errors in Step 4 before deciding to keep the change.

## Step 4 — Analyze mistakes and fix annotations

Where does the model still get it wrong? Before optimizing further, it is worth separating two types of remaining errors:

- **Model errors** — the prompt and label definitions are clear, but the model still misclassifies.
- **Label errors** — the gold label in the dataset is ambiguous or incorrect; the model's answer may actually be right.

This step identifies both. High-confidence errors are the best place to start: if the model was confident and wrong, either the prompt needs sharper label definitions, or the gold label is the problem.

In [ ]:
mistake_examples = await prepare_mistake_examples(
    eval_df=df,
    predictions=improved_prompt_run.predictions,
    max_per_type=3,
)

display(mistake_examples.head(10))

Scan for patterns, not individual cases.

- If the same two departments appear repeatedly on opposite sides of an error (e.g. Returns predicted as Customer Support), the prompt's label definitions for those two classes are not distinct enough for the model to separate them.
- If errors are spread randomly across many departments, the issue is more general — the prompt may be under-specified overall.
- A small number of isolated errors on unusual cases is normal and does not require action at this stage.

#### High-confidence errors (calibration check)

High-confidence errors are the most important diagnostic in this notebook. If this set is large (more than 10–15% of all High-confidence predictions), the model's confidence signal is not calibrated — it is overconfident. Auto-routing on uncalibrated confidence creates the exact risk the safety guardrail was designed to prevent.

For each error, ask: is the model clearly wrong, or is the gold label ambiguous? Some of these cases will be data quality issues, not model failures. The re-annotation step below separates the two.

In [ ]:
high_conf_errors_df = display_high_confident_errors(
    eval_df=df,
    predictions=improved_prompt_run.predictions,
    max_rows=10,
)

#### Re-annotation candidates (top 10)

Export 10 high-confidence errors to `fixed-annotation.json` for manual review.
This creates a small queue of examples to verify/correct gold labels before re-testing.

- Allowed Department: Logistics, Customer Support, Product Team, Returns
- Allowed Category: Delivery, Order Issue, General Feedback, Payment

In [ ]:
analyzed_high_conf_df = analyze_high_confident_errors(high_conf_errors_df)

llm_review_df = await recommend_new_labels(
    analyzed_high_conf_df,
    client=client,
    model=MODEL_TO_EVALUATE,
)

revise_df = llm_review_df.loc[llm_review_df["Revise labels"]].head(10)
print(f"LLM relabel recommendations analyzed: {len(llm_review_df)}")
print(f"Rows flagged for revision: {int(llm_review_df['Revise labels'].sum())}")
display(revise_df.head(3))

#### Apply label fixes

Based on the review above, correct the gold labels for ambiguous or mis-annotated rows, then re-evaluate to see if the fixes improve results.

In [ ]:
revised_labels = [
    {"row_id": 4, "new_category": "General Feedback"},
    {"row_id": 13, "new_category": "Payment"},
    {"row_id": 26, "new_department": "Returns"},
    {"row_id": 33, "new_category": "General Feedback", "new_department": "Product Team"},
    {"row_id": 40, "new_category": "Order Issue", "new_department": "Returns"},
    {"row_id": 57, "new_category": "General Feedback", "new_department": "Product Team"},
    {"row_id": 65, "new_category": "General Feedback"},
    {"row_id": 83, "new_category": "General Feedback", "new_department": "Product Team"},
    {"row_id": 88, "new_category": "Delivery", "new_department": "Logistics"},
    {"row_id": 90, "new_category": "General Feedback", "new_department": "Product Team"},
    {"row_id": 94, "new_department": "Logistics"},
    {"row_id": 95, "new_category": "Delivery", "new_department": "Logistics"},
    {"row_id": 99, "new_department": "Logistics"},
]

# Apply recommended labels
df = apply_revised_labels(df, revised_labels)

# Check that the labels were updated correctly
df.loc[df["row_id"] == 4, "Category"]

In [ ]:
revised_labels_run = await evaluate_model_on_dataframe_async(
    df=df,
    model=MODEL_TO_EVALUATE,
    client=client,
    system_prompt=IMPROVED_SYSTEM_PROMPT,
)

In [ ]:
revised_labels_decision = evaluate_decision(revised_labels_run)
display_mvp_decision(decision=revised_labels_decision)

In [ ]:
_ = display_run_comparison_table(
    baseline=improved_prompt_run,
    new=revised_labels_run,
    eval_df=df,
    new_label="Fix data annotation",
    show_all=False,
)

An accuracy gain here confirms that some previous "errors" were label noise, not model failures. This matters beyond this notebook: better labels improve every downstream approach — prompt tuning, few-shot examples, and fine-tuning all benefit from cleaner ground truth.

If accuracy did not improve after label fixes, the remaining errors are genuine model mistakes. Focus Notebook B iteration on prompt clarity and few-shot examples rather than further annotation work.

## Step 5 — Few-shot trade-offs

Do labeled examples in the prompt help? We run the same model with increasing numbers of few-shot examples and compare quality, cost, and latency.

The ablation runs on top of the improved prompt from Step 3, so the selected example count reflects the gain few-shot adds to the best prompt found so far — not the baseline prompt.

The goal is to find the point where adding more examples **stops helping** (diminishing returns) relative to the cost and latency increase.

In [ ]:
N_SHOT_VALUES = [0, 2, 4, 8, 16]

ablation_config = FewShotAblationConfig(
    best_model=MODEL_TO_EVALUATE,
    few_shot_counts=N_SHOT_VALUES,
    ablation_random_state=42,
    few_shot_pool_n=32,
    ablation_eval_n=68,
)

few_shot_pool_df, eval_df_fixed, few_shot_counts = prepare_few_shot_ablation_slice(
    df,
    config=ablation_config,
)

# Measure few-shot gain on top of the improved prompt (same prompt as best_n_run),
# so the selected n_shot is comparable to the rest of this notebook.
ablation_df, _phase2_predictions = await run_few_shot_ablation_async(
    few_shot_pool_df=few_shot_pool_df,
    eval_df_fixed=eval_df_fixed,
    few_shot_counts=few_shot_counts,
    best_model=MODEL_TO_EVALUATE,
    client=client,
    system_prompt=IMPROVED_SYSTEM_PROMPT,
    show_progress=True,
)

ablation_default_columns = [
    "n_shot",
    "department_accuracy",
    "category_accuracy",
    "cost_total_usd",
    "p95_latency_ms",
]
display(ablation_df[[c for c in ablation_default_columns if c in ablation_df.columns]])

### Visualize few-shot trade-offs

The charts below show how accuracy, cost, and latency change as we add more examples.

In [ ]:
_ = plot_metrics_for_n_shot(ablation_df, best_model=MODEL_TO_EVALUATE)

The accuracy curve shows the quality return on adding examples.

- If accuracy rises steeply from 0 to 2–4 examples and then flattens, 4 examples is probably the right operating point.
- If the curve rises continuously through 8–16 examples, the model is benefiting from more demonstration — but check the cost chart before committing to a high n_shot value.
- A curve that does not improve beyond 0-shot means the prompt is already specific enough and few-shot examples are not adding information.

In [ ]:
_ = plot_latency_for_n_shots(ablation_df, best_model=MODEL_TO_EVALUATE)

Each additional example adds tokens to every request, which increases both cost and latency.

- Compare the latency at your candidate n_shot value against the Step 8 threshold from Notebook A (p95 <5,000 ms). If adding examples pushes p95 above that threshold, the quality gain may not be worth it.
- Cost grows roughly linearly with n_shot. Calculate the monthly cost delta at your candidate n_shot value and check it against the H6 budget before selecting.

### Run with the best few-shot count

Based on the charts above, pick the `n_shot` value with the best accuracy-to-cost balance and run a full evaluation.

In [ ]:
# Choose best n_shot from ablation (priority: department accuracy, then category accuracy).
best_n = 8
print(f"Selected best n_shot = {best_n}")

In [ ]:
best_n_run = await evaluate_model_on_dataframe_async(
    df=df,
    model=MODEL_TO_EVALUATE,
    client=client,
    system_prompt=IMPROVED_SYSTEM_PROMPT,
    few_shot_df=few_shot_pool_df.head(best_n),
)

In [ ]:
best_n_decision = evaluate_decision(best_n_run)
display_mvp_decision(decision=best_n_decision)

In [ ]:
# Compare best n-shot with revised labels run

_ = display_run_comparison_table(
    baseline=revised_labels_run,
    new=best_n_run,
    eval_df=df,
    new_label=f"Few-shot best n={best_n}",
    show_all=False,
)

### Deeper analysis

What mistakes remain after all improvements? The diagnostics below show recurring department confusions to guide further iteration.

In [ ]:
mistakes = display_evaluation_with_department_mistakes(
    eval_results=best_n_run,
    eval_df=df,
    max_rows=10,
)

If the same department confusions recur here that you saw in Step 4, further iteration on label definitions is the most direct path to improvement. If errors are different from Step 4, the few-shot examples may be introducing new confusion.

## Summary

This notebook tested three practical levers against the Notebook A baseline:

1. **Prompt improvement** — clearer instructions reduce label confusion and improve routing accuracy.
2. **Data quality** — fixing ambiguous gold labels removes noise and gives a more accurate picture of true model performance.
3. **Few-shot examples** — labeled examples in the prompt can boost accuracy, but add cost and latency that must be checked against thresholds.

These levers are not exclusive. The best configuration usually combines all three.

**Record your final results before moving to the workflow module:**

| Dimension | Notebook A baseline | Best result here | Change | Pass threshold |
|-----------|--------------------|--------------------|--------|----------------|
| Department accuracy | | | | ≥85% |
| Misroute rate | | | | ≤10% |
| Unsafe auto-route rate | | | | ≤3% |
| Monthly cost | | | | <$1,000 |
| p95 latency | | | | <5,000 ms |

If all dimensions now pass, the configuration is ready for workflow integration. If one dimension still fails, decide whether further iteration is warranted or whether the threshold should be revisited given what you learned about the data.